In [10]:
import pandas as pd

df = pd.read_parquet("data/laion-art.parquet")
df.head()

,URL,TEXT,WIDTH,HEIGHT,similarity,LANGUAGE,hash,pwatermark,punsafe,aesthetic
0,https://www.advocate-art.com/system/ART/Module...,Christmas Shopping Copy,850.0,850.0,0.266625,nolang,-3604776403351267688,0.039626,0.000278,8.352225
1,https://img.freepik.com/fotos-kostenlos/cremig...,Cremiger hüttenkäse im tontopf und kekse,626.0,415.0,0.277981,de,571058205933102479,0.064438,0.001128,8.013723
2,https://zakarpat.brovdi.art/media/zoo/images/4...,A. Landovska Breakfast',300.0,344.0,0.281941,sv,-8455279219565229056,0.023449,0.000022,8.043530
3,https://www.povarenok.ru/data/cache/2016oct/16...,Рецепт: Воздушные пончики Творожные колечки,330.0,220.0,0.260818,ru,-3137830403123787972,0.337415,0.004736,8.149608
4,http://media.independent.com/img/photos/2017/0...,"For nearly a decade, Slightly Stoopid (picture...",320.0,200.0,0.377336,en,-7379866731095810216,0.065501,0.014704,9.041004


In [ ]:
import os
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

os.makedirs("src/data/laion-art_img", exist_ok= True)
sample_df = df.sample(3000, random_state= 42)
def download_image(row_data):
    idx, row = row_data
    try:
        url = row["URL"]
        caption = row["TEXT"]
        response = requests.get(url, timeout= 5)

        image = Image.open(BytesIO(response.content)).convert("RGB")
        image.save(f"src/data/laion-art_img/{idx}.jpg")

        with open(f"src/data/laion-art_img/{idx}.txt", "w", encoding= "utf-8") as f:
            f.write(caption)

    except Exception:
        pass

rows = list(sample_df.iterrows())
with ThreadPoolExecutor(max_workers=32) as executor:
    list(tqdm(executor.map(download_image, rows), total= len(rows)))

C:\Users\ENG_AHMED_AYAD\anaconda3\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 3000/3000 [01:38<00:00, 30.36it/s]


In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

class CrossAttention(nn.Module):
    def __init__(self, d_model, context_dim, heads=8):
        super().__init__()
        assert d_model % heads == 0
        self.heads    = heads
        self.head_dim = d_model // heads
        self.scale    = self.head_dim ** -0.5

        self.to_q   = nn.Linear(d_model,     d_model,     bias=False)
        self.to_k   = nn.Linear(context_dim, d_model,     bias=False)
        self.to_v   = nn.Linear(context_dim, d_model,     bias=False)
        self.to_out = nn.Linear(d_model,     d_model)

    def forward(self, x, context):
        b, c, h, w = x.shape
        x_flat = x.view(b, c, h * w).permute(0, 2, 1)  # (B, H*W, C)

        q = self.to_q(x_flat)   # (B, H*W,    d_model)
        k = self.to_k(context)  # (B, seq_len, d_model)
        v = self.to_v(context)  # (B, seq_len, d_model)

        # split heads — each tensor may have a different seq length
        def split_heads(t):
            bsz, seq, _ = t.shape
            return t.view(bsz, seq, self.heads, self.head_dim).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # (B, heads, H*W, seq_len)
        attn = F.softmax(attn, dim=-1)

        out = torch.matmul(attn, v)                               # (B, heads, H*W, head_dim)
        out = out.transpose(1, 2).contiguous().view(b, h * w, c)  # (B, H*W, C)
        out = self.to_out(out)
        return out.permute(0, 2, 1).view(b, c, h, w)

class UNetConditional(nn.Module):
    def __init__(self, latent_channels=4, context_dim=768):
        super().__init__()

        self.time_mlp = nn.Sequential(
            nn.Linear(1, 256),
            nn.SiLU(),
            nn.Linear(256, 256),
        )
        self.time_proj1 = nn.Linear(256, 128)
        self.time_proj2 = nn.Linear(256, 256)

        self.down1 = nn.Conv2d(latent_channels, 128, kernel_size=3, padding=1)    
        self.attn1 = CrossAttention(d_model=128, context_dim=context_dim)

        self.down2 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)   
        self.attn2 = CrossAttention(d_model=256, context_dim=context_dim)

        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)  
        
        self.conv_merge = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.out = nn.Conv2d(128, latent_channels, kernel_size=3, padding=1)

    def forward(self, x, t, context):
        t_emb  = self.time_mlp(t.float().unsqueeze(-1))      
        t_emb1 = self.time_proj1(t_emb).unsqueeze(-1).unsqueeze(-1)   
        t_emb2 = self.time_proj2(t_emb).unsqueeze(-1).unsqueeze(-1)   

        h1 = F.silu(self.down1(x) + t_emb1)   
        h1 = self.attn1(h1, context)   

        h2 = F.silu(self.down2(h1) + t_emb2)
        h2 = self.attn2(h2, context)

        h_up = F.silu(self.up1(h2))
        
        h_skip = torch.cat([h_up, h1], dim=1)
        h_skip = F.silu(self.conv_merge(h_skip))

        return self.out(h_skip)

In [22]:
from torch.utils.data import Dataset
import torchvision.transforms as transforms

class ImageCaptionDataset(Dataset):
    def __init__(self, image_folder, tokenizer):
        self.image_folder = image_folder
        self.tokenizer = tokenizer
        self.images = [f for f in os.listdir(image_folder) if f.endswith(".jpg")]
        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        image = Image.open(os.path.join(self.image_folder, img_name)).convert("RGB")
        image = self.transform(image)

        txt_path = os.path.join(self.image_folder, img_name.replace(".jpg", ".txt"))
        with open(txt_path, "r", encoding="utf-8") as f:
            caption = f.read()

        tokens = self.tokenizer(caption, padding="max_length", truncation=True, max_length=77, return_tensors="pt")
        
        return {
            "pixel_values": image,
            "input_ids": tokens.input_ids.squeeze(0)
        }

In [26]:
from torch.utils.data import DataLoader
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, DDPMScheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

dataset = ImageCaptionDataset(image_folder="data/laion-art_img", tokenizer=tokenizer)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
text_encoder.eval()

vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()

noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

model = UNetConditional(context_dim=512).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.embeddings.position_ids                             | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.

In [27]:
epochs = 10
model.train()

for epoch in range(epochs):
    print(f"Starting Epoch {epoch+1}/{epochs}")
    for batch in dataloader:
        images = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)

        with torch.no_grad():
            context = text_encoder(input_ids).last_hidden_state

        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample()
            latents = latents * 0.18215

        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=device).long()
        
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        noise_pred = model(noisy_latents, timesteps, context)
        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} Complete. Loss: {loss.item():.4f}")

Starting Epoch 1/10


KeyboardInterrupt: 

In [18]:
batch = next(iter(dataloader))

print(batch["pixel_values"].shape)
print(batch["input_ids"].shape)

images = batch["pixel_values"].to(device)
with torch.no_grad():
    latents = vae.encode(images).latent_dist.sample()
print(latents.shape)


noise_pred = model(latents, torch.randint(0, 1000, (4,), device= device), context)
print(noise_pred.shape)

torch.Size([4, 3, 256, 256])
torch.Size([4, 77])
torch.Size([4, 4, 32, 32])
torch.Size([4, 4, 32, 32])
